## 2.6 Notification System

Batch runner for budget threshold alerts and personalized tips.

Core logic: `model/notify_core.py`

**Alerts**
- Budget thresholds at **70% / 85% / 95%** MTD discretionary utilization
- Overspend / short-runway warnings from the prediction artifact
- Personalized tip from the top recommendation

**Channels**
- `app` — rendered in Streamlit
- `email` — same payload recorded for multi-channel delivery (no live SMTP in v1)

**Output:** `artifacts/events_{CLIENT_ID}.jsonl`


## Imports

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir() and (candidate / "artifacts").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/ and artifacts/")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model.analytics_core import load_transactions_for_client, project_root
from model.notify_core import run_notifications_for_client
from model.recommend_core import generate_recommendations


## Config

In [ ]:
CLIENT_ID = 1696
AS_OF_DATE: str | None = None  # e.g. "2018-02-28"; default = budget artifact / latest

root = project_root()
art = root / "artifacts"
print("ROOT:", root)
print("CLIENT_ID:", CLIENT_ID)


## Generate notifications

In [ ]:
budget_path = art / f"budget_utilization_{CLIENT_ID}.json"
runway_path = art / f"runway_{CLIENT_ID}.json"

budget = json.loads(budget_path.read_text()) if budget_path.exists() else {}
prediction = json.loads(runway_path.read_text()) if runway_path.exists() else {}

as_of = AS_OF_DATE or budget.get("as_of_date") or prediction.get("as_of_date")
if as_of is None:
    raise FileNotFoundError(
        "Need budget_utilization / runway artifacts (or set AS_OF_DATE). "
        "Run analytics + predict first."
    )

tx = load_transactions_for_client(CLIENT_ID, root=root)
recs = generate_recommendations(
    tx,
    client_id=CLIENT_ID,
    as_of_date=pd.to_datetime(as_of).date(),
    monthly_limit_usd=budget.get("monthly_discretionary_limit_usd")
    or prediction.get("monthly_discretionary_limit_usd"),
    root=root,
    max_recommendations=2,
)

result = run_notifications_for_client(
    CLIENT_ID,
    root=root,
    as_of_date=as_of,
    budget=budget,
    prediction=prediction,
    recommendations=recs,
    write_artifact=True,
)

print("Wrote:", result["events_path"])
print("Event count:", len(result["events"]))
print("App channel:", len(result["app_events"]))
print("Email channel:", len(result["email_events"]))
for ev in result["events"]:
    print(f"- [{ev.severity}/{ev.kind}] {ev.title}")
    print(f"  channels={ev.channels}")
    print(f"  {ev.message}")
